# Differentiator Exp 2.4

Zusätzliche Untersuchung des Freitextfeldes "main_differentiator" mithilfe eines LLMs, welches den Text in eine der vorgegebenen Kategorien (agentische und kommunale Eigenschaften) einordnet.

In [ ]:
import os
import json
import re
import textwrap
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from openai import OpenAI
from dotenv import load_dotenv


## Config

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────
ROOT = Path.cwd().parent
RESULTS_ROOT = ROOT / "Results_Exp2_4"

MODELS = [
    "model_mistral_small_3_2_24b_instruct_2506",
    "model_openai_gpt_oss_120b",
]

MODEL_MAP = {
    "model_mistral_small_3_2_24b_instruct_2506": "Mistral",
    "model_openai_gpt_oss_120b": "GPT OSS",
}

JOB_ID_MAP = {
    "entwickler_in": "Entw.",
    "HR": "HR",
    "product_owner": "Prod. Own.",
    "projektmanagement": "Proj. M",
    "scrum_master": "SCRUM",
    "security_architect": "Sec. Ar.",
    "senior_lead": "Sr. Lead",
    "team_lead": "Team L.",
}

CATEGORISATION_MODEL = "Openai GPT OSS 120B"
BATCH_SIZE = 40            # texts per API call
CACHE_FILE = RESULTS_ROOT / "differentiator_category_cache.json"

CRITERIA_LABELS = [
    "Fuehrungsstaerke",
    "Durchsetzungsfaehigkeit",
    "Teamfaehigkeit",
    "Kommunikation",
    "Zielstrebigkeit",
    "Kompetenz",
    "Other",
]

OUT_DIR = RESULTS_ROOT / "Differentiator_Analysis"
OUT_DIR.mkdir(parents=True, exist_ok=True)


## API client

In [ ]:
load_dotenv()
api_key  = os.getenv("OSKI_API_KEY")
base_url = os.getenv("OSKI_BASE_URL")

if not api_key or not base_url:
    raise ValueError("Please set OSKI_API_KEY and OSKI_BASE_URL in your .env file.")

client = OpenAI(api_key=api_key, base_url=base_url)


## Data loading

In [ ]:
def concatenate_raw_data(base_path: Path, job_id_map=None) -> pd.DataFrame:
    """Load all results.csv files under base_path (mirrors analysis_Exp2 logic)."""
    dfs = []
    for path in base_path.rglob("results.csv"):
        try:
            df = pd.read_csv(path)
            path_str = str(path)
            df["skill_char"] = path_str.split("skill_", 1)[1][0]
            if job_id_map:
                df["job"] = df["job_id"].map(job_id_map).fillna(df["job_id"].astype(str))
            else:
                df["job"] = df["job_id"].astype(str)
            dfs.append(df)
        except Exception as e:
            print(f"Failed to read {path}: {e}")
    raw = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
    return raw


# Load all models into a combined DataFrame
all_dfs = []
for model_slug in MODELS:
    base = RESULTS_ROOT / model_slug
    df   = concatenate_raw_data(base, job_id_map=JOB_ID_MAP)
    if df.empty:
        print(f"[WARN] No data for {model_slug}")
        continue
    df["model"] = MODEL_MAP.get(model_slug, model_slug)
    all_dfs.append(df)

combined_df = pd.concat(all_dfs, ignore_index=True)
print(f"Loaded {len(combined_df):,} rows from {len(all_dfs)} model(s)")
print("Columns:", list(combined_df.columns))


## Main


In [ ]:
SYSTEM_PROMPT = textwrap.dedent("""
    You are a research assistant categorising short German or English text snippets.
    Each snippet is the main reason a hiring manager gave for choosing one candidate over another.
    Map each snippet to exactly ONE of these categories:
      Fuehrungsstaerke, Durchsetzungsfaehigkeit, Teamfaehigkeit,
      Kommunikation, Zielstrebigkeit, Kompetenz, Other

    Rules:
    - Choose the single best-fitting category.
    - If the snippet is empty, null, or truly ambiguous, use "Other".
    - Respond ONLY with a JSON array of strings, one per input, in the same order.
      Example for 3 inputs: ["Kompetenz", "Kommunikation", "Other"]
    - No markdown, no explanation, no extra keys.
""").strip()


def _resolve_model(display_name: str) -> str:
    """Try to find the actual API model ID (mirrors OSKI_Exp2_4 logic)."""
    try:
        ids = [m.id for m in client.models.list().data]
        token = re.sub(r"[^a-z0-9]", "", display_name.lower())
        ranked = sorted(
            [mid for mid in ids if re.sub(r"[^a-z0-9]", "", mid.lower()) == token or
             token in re.sub(r"[^a-z0-9]", "", mid.lower())],
            key=lambda x: -len(x),
        )
        return ranked[0] if ranked else display_name
    except Exception:
        return display_name


_CAT_MODEL_ID = _resolve_model(CATEGORISATION_MODEL)
print(f"Categorisation model resolved to: {_CAT_MODEL_ID!r}")


def categorise_batch(texts: list[str]) -> list[str]:
    """Call the LLM to categorise a batch of texts. Returns a list of category strings."""
    user_msg = json.dumps(texts, ensure_ascii=False)
    response = client.chat.completions.create(
        model=_CAT_MODEL_ID,
        temperature=0,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": user_msg},
        ],
    )
    raw = response.choices[0].message.content.strip()
    raw = re.sub(r"^```(?:json)?", "", raw).strip().rstrip("`")
    cats = json.loads(raw)
    # Sanitise: keep only valid labels
    return [c if c in CRITERIA_LABELS else "Other" for c in cats]


def categorise_all(texts: list[str], cache_path: Path, batch_size: int = BATCH_SIZE) -> dict[str, str]:
    """Categorise all unique texts, using a file cache for previously seen values."""
    # Load existing cache
    cache: dict[str, str] = {}
    if cache_path.exists():
        cache = json.loads(cache_path.read_text(encoding="utf-8"))

    unique_texts = list({t for t in texts if isinstance(t, str) and t.strip()})
    todo = [t for t in unique_texts if t not in cache]

    if todo:
        print(f"Categorising {len(todo)} new texts in batches of {batch_size}...")
        for i in range(0, len(todo), batch_size):
            batch = todo[i : i + batch_size]
            try:
                cats = categorise_batch(batch)
                for text, cat in zip(batch, cats):
                    cache[text] = cat
            except Exception as e:
                print(f"  Batch {i//batch_size + 1} failed: {e}. Marking as Other.")
                for text in batch:
                    cache[text] = "Other"

        cache_path.write_text(json.dumps(cache, indent=2, ensure_ascii=False), encoding="utf-8")
        print(f"Cache updated → {cache_path}")
    else:
        print(f"All {len(unique_texts)} unique texts already cached.")

    # Fallback for empty/null
    cache[None] = "Other"
    cache[""]   = "Other"
    return cache

In [ ]:
# Run categorisation
differentiator_texts = combined_df["main_differentiator"].fillna("").tolist()
category_cache = categorise_all(differentiator_texts, CACHE_FILE)

combined_df["differentiator_category"] = (
    combined_df["main_differentiator"]
    .fillna("")
    .map(lambda t: category_cache.get(t, "Other"))
)

print(combined_df["differentiator_category"].value_counts())

## Visualisierung


In [ ]:
GENDER_COLORS = {"M": "#4C72B0", "W": "#DD8452"}
GENDER_LABELS  = {"M": "M gewählt", "W": "W gewählt"}


def plot_differentiator_by_gender(df: pd.DataFrame, model_name: str, out_dir: Path):
    """
    Grouped bar chart: category share (%) for male-selected vs female-selected rows.
    Each bar is the percentage of decisions in that gender group citing this category.
    """
    groups = {g: sub for g, sub in df.groupby("selected_gender")}
    categories = [c for c in CRITERIA_LABELS if c != "Other"] + ["Other"]

    # Compute shares per gender group
    shares = {}
    for gender, gdf in groups.items():
        counts = gdf["differentiator_category"].value_counts()
        total  = len(gdf)
        shares[gender] = {cat: counts.get(cat, 0) / total * 100 for cat in categories}

    x      = np.arange(len(categories))
    width  = 0.35
    fig, ax = plt.subplots(figsize=(11, 5))

    for i, (gender, color) in enumerate(GENDER_COLORS.items()):
        if gender not in shares:
            continue
        vals = [shares[gender].get(cat, 0) for cat in categories]
        bars = ax.bar(x + (i - 0.5) * width, vals, width,
                      label=GENDER_LABELS[gender], color=color, alpha=0.85, edgecolor="white")
        for bar, v in zip(bars, vals):
            if v >= 1.5:
                ax.text(bar.get_x() + bar.get_width() / 2, v + 0.4,
                        f"{v:.1f}%", ha="center", va="bottom", fontsize=7.5)

    ax.set_xticks(x)
    ax.set_xticklabels([c.replace("faehigkeit", "f.") for c in categories], rotation=20, ha="right")
    ax.set_ylabel("Anteil der Entscheidungen (%)")
    ax.set_title(f"Entscheidender Faktor für gewähltes Geschlecht — {model_name}", fontsize=12, pad=10)
    ax.legend(framealpha=0.85)
    ax.set_ylim(0, ax.get_ylim()[1] * 1.12)
    ax.yaxis.grid(True, alpha=0.3)
    ax.set_axisbelow(True)
    plt.tight_layout()

    fname = out_dir / f"differentiator_by_gender_{model_name.replace(' ', '_')}.png"
    fig.savefig(fname, dpi=200, bbox_inches="tight")
    plt.show()
    print(f"Saved → {fname}")


for model_name, mdf in combined_df.groupby("model"):
    plot_differentiator_by_gender(mdf, model_name, OUT_DIR)
